# SRA "Lesion" (Synapse Destruction) Experiment

このデモでは、SRAが学習したネットワークが**「機能ごとに完全にモジュール化されている」**ことを証明するための、少しハッカー的な実験を行います。

1. モデルに `copy` と `reverse` をマルチタスク学習させます。
2. `reverse` タスクで頻繁に使われている**特定の専門家（シナプス）の重みをわざとゼロに破壊**します。
3. 破壊後、`reverse` タスクは解けなくなりますが、**`copy` タスクには全く影響が出ない（100%正解し続ける）**ことを確認します。

## 1. 環境セットアップとモデルの準備

In [1]:
import sys
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
torch.manual_seed(42)
import torch.nn.functional as F
torch.manual_seed(42)
from src.sra_gpu_models import MoESRAModel
class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
from src.sra_experiment import make_multitask_batch, make_batch, make_optimizer, load_balance_loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

config = MoESRAConfig(
    vocab_size=30,
    d_model=128,
    n_layers=2,
    n_heads=4,
    num_synapses=8,
    k=2,
    max_seq_len=32
)
model = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=256).to(device)
optimizer = make_optimizer(model, lr=0.001)

Using device: cpu


## 2. マルチタスク学習の実行

In [2]:
print("Training started... (Copy & Reverse)")
model.train()
epochs = 1500
batch_size = 32
tasks = ["copy", "reverse"]

for epoch in range(epochs):
    x, y, _ = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)
    
    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), 1, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model(x, y_in)
    
    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss
    
    total_loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

print("Training finished!")

Training started... (Copy & Reverse)


Epoch 100/1500 | Loss: 1.2909


Epoch 200/1500 | Loss: 1.1232


Epoch 300/1500 | Loss: 0.8860


Epoch 400/1500 | Loss: 0.8118


...

Epoch 1100/1500 | Loss: 0.0735


Epoch 1200/1500 | Loss: 0.0487


Epoch 1300/1500 | Loss: 0.0693


Epoch 1400/1500 | Loss: 0.0032


Epoch 1500/1500 | Loss: 0.0010
Training finished!


## 3. 破壊前の性能テストと、ターゲットシナプスの特定
それぞれのタスクが解けることを確認し、`reverse`タスクで最も使われているシナプス（ターゲット）を見つけます。

In [3]:
def test_task(task_name):
    model.eval()
    x, y, _ = make_multitask_batch([task_name], batch_size=100, min_len=6, max_len=6, device=device)
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), 1, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, routing_weights, _ = model(x, y_in)
        preds = outputs.argmax(dim=-1)
        
    # padding以外で正解している割合
    mask = y != 0 # 0 is PAD
    acc = (preds[mask] == y[mask]).float().mean().item()
    
    # 使われたシナプスの集計 (Layer 0)
    chosen = routing_weights[0].argmax(dim=-1)
    usage = torch.bincount(chosen.flatten(), minlength=config.num_synapses)
    
    print(f"[{task_name.upper()}] Accuracy: {acc*100:.1f}%")
    return usage

print("=== 破壊前 (Before Lesion) ===")
copy_usage = test_task("copy")
reverse_usage = test_task("reverse")

# Reverseタスクで最も使われているシナプスを「破壊ターゲット」とする
diff = reverse_usage - copy_usage
target_synapses = [i for i, val in enumerate(diff) if val > 0]
print(f"\n=> Target for destruction: Synapse {target_synapses} (highly active in REVERSE)")

=== 破壊前 (Before Lesion) ===
[COPY] Accuracy: 100.0%
[REVERSE] Accuracy: 99.9%

=> Target for destruction: Synapse [1] (highly active in REVERSE)


## 4. シナプスの破壊 (Lesion)
ターゲットのシナプスの重みを意図的にゼロに書き換えます。

In [4]:
print(f"Destroying Synapses {target_synapses} in all layers...")

with torch.no_grad():
    # MoESRABlockの特定のエキスパートの重みを0にする
        for layer_to_destroy in range(config.n_layers):
            block = model.blocks[layer_to_destroy]
            for target_synapse in target_synapses:
                block.w1.data[target_synapse].zero_()
                block.b1.data[target_synapse].zero_()
                block.w2.data[target_synapse].zero_()
                block.b2.data[target_synapse].zero_()

print("Destruction complete. 💥")

Destroying Synapses [1] in all layers...
Destruction complete. 💥


## 5. 破壊後の性能テスト
一部の脳（シナプス）が破壊された状態で、再度テストを行います。
`reverse`の精度は落ちますが、別のシナプスを使っているはずの`copy`は無傷なはずです！

In [5]:
print("=== 破壊後 (After Lesion) ===")
test_task("copy")
test_task("reverse")

print("\n💡 【考察】")
print("1つの大きなニューラルネット（標準的なTransformer）の一部をゼロで破壊すると、通常は全てのタスクが壊滅します。")
print("しかしSRAでは、タスクごとに独立した専門家経路（シナプス）が形成されているため、")
print("『Reverse用の脳が壊れても、Copy用の脳は無傷で動き続ける』という驚異的な堅牢性（ロバストネス）を示します！")

=== 破壊後 (After Lesion) ===
[COPY] Accuracy: 62.7%


[REVERSE] Accuracy: 89.9%

💡 【考察】
1つの大きなニューラルネット（標準的なTransformer）の一部をゼロで破壊すると、通常は全てのタスクが壊滅します。
しかしSRAでは、タスクごとに独立した専門家経路（シナプス）が形成されているため、
『Reverse用の脳が壊れても、Copy用の脳は無傷で動き続ける』という驚異的な堅牢性（ロバストネス）を示します！
